<a href="https://colab.research.google.com/github/matti410/Trading-System-Creator_V4/blob/main/Collaudo_Catalogo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Collaudo del tool · lookahead, degenerazione, backtesting.py, soglie di rumore

Questo notebook controlla **tutte le condizioni registrate** (entry, exit, filtri) e gli indicatori.

- **Sezione A** — collaudo del collaudo: condizioni-trappola costruite apposta. Se A non passa, B e C non valgono.
- **Sezione B** — lookahead su tutto il catalogo, su dati sintetici. Esito: promosso / bocciato.
- **Sezione C** — degenerazione sui dati veri (EURUSD dal repo). Esito: una tabella da leggere.
- **Sezione D** — i tre comportamenti di `backtesting.py` su cui si regge il tool. Esito: promosso / bocciato.
- **Sezione E** — le soglie di rumore: su entry casuali devono scattare per sbaglio circa 5 volte su 100. Esito: promosso / bocciato.

**Come si esegue:** in Colab, menu *Runtime → Esegui tutto*. Tempo totale: 6–8 minuti, quasi tutto nelle sezioni B ed E.

Il notebook clona il repo da GitHub: i file `engine/indicatori.py` e `engine/collaudo_catalogo.py` devono essere già caricati sul repo.

## 0 · Preparazione

Scarica il repo (o lo aggiorna, se c'è già) e installa **le versioni bloccate** di TA-Lib e backtesting.py, le stesse del notebook principale. Alla fine stampa le versioni: se qualcosa va storto, sono le prime cose da guardare.

Le versioni si cambiano **solo** qui e nella cella 2 del notebook principale, insieme, e dopo si rilancia questo notebook: la sezione D dice se il tool regge ancora.

In [ ]:
import os
REPO = "Trading-System-Creator_V4"
if not os.path.exists(REPO):
    !git clone -q https://github.com/matti410/Trading-System-Creator_V4.git
else:
    !git -C {REPO} pull -q
%cd {REPO}
!pip install -q TA-Lib==0.8.1 backtesting==0.6.6

import sys, numpy as np, pandas as pd, talib, backtesting
print(f"python {sys.version.split()[0]} · pandas {pd.__version__} · numpy {np.__version__} · "
      f"ta-lib {talib.__version__} · backtesting {backtesting.__version__}")

## 1 · Il catalogo

`registra_catalogo()` chiama tutte le funzioni `registra_*`. **Una condizione nuova in un file esistente è coperta da sola.** Un *file* di condizioni nuovo va aggiunto con una riga alla lista `REGISTRAZIONI`.

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import contextlib, io
from engine import registry as R
from engine.indicatori import aggiungi_indicatori
from engine.collaudo_catalogo import (condizioni_registrate, verifica_lookahead,
                                      verifica_degenerazione, mercato_sintetico)
from engine.broker_tz_diagnostic import to_utc_index
from helpers import _evento
import entry_long, entry_short, entry_metro, exit_long, exit_short
import filter_conditions, vwap_regime_filter_conditions

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 160)

REGISTRAZIONI = [
    entry_long.registra_trigger_long,
    entry_short.registra_trigger_short,
    lambda: entry_metro.registra_trigger_metro(anche_short=True),
    exit_long.registra_exit_long,
    exit_short.registra_exit_short,
    filter_conditions.registra_filtri,
    vwap_regime_filter_conditions.registra_filtri_vwap,
]

def registra_catalogo():
    R.clear_registry()
    with contextlib.redirect_stdout(io.StringIO()):   # niente righe di log
        for f in REGISTRAZIONI:
            f()

registra_catalogo()
catalogo = condizioni_registrate()
print(f"{len(catalogo)} condizioni registrate")
catalogo["tipo"].value_counts()

## A1 · Collaudo del test di lookahead

Si registrano condizioni-trappola **con lookahead messo apposta**, due condizioni pulite e una mai vera. Il test deve riconoscerle tutte. C'è anche un indicatore-trappola (`media_centrata`), per verificare che il test ricalcoli gli indicatori a ogni taglio.

In fondo: `SEZIONE A1: n/n`. Se non sono tutte giuste, il notebook si ferma.

In [ ]:
R.clear_registry()

# --- trappole: ognuna guarda avanti in un modo diverso -------------------
def t_shift_meno_uno(df):      # legge la chiusura della barra dopo
    return _evento(df["Close"].shift(-1) > df["Close"])

def t_shift_raro(df):          # come sopra, ma rara: serve ai tagli mirati
    salto = df["Close"].shift(-1) - df["Close"]
    return _evento(salto > 3 * salto.abs().rolling(500).mean())

def t_massimo_giorno(df):      # il massimo del giorno si conosce solo a fine giorno
    return df["High"] == df["High"].groupby(df.index.date).transform("max")

def t_rolling_centrata(df):    # media centrata: usa 4 barre future
    return df["Close"] > df["Close"].rolling(9, center=True).mean()

def t_quantile_globale(df):    # soglia calcolata su tutto lo storico
    return df["atr"] > df["atr"].quantile(0.7)

def t_prossimo_timestamp(df):  # "la barra dopo non c'e'": ultima prima del weekend
    tempi = pd.Series(df.index, index=df.index)
    return tempi.diff().shift(-1) > pd.Timedelta(hours=1)

_CACHE_TRAPPOLA = {}
def t_cache(df):               # lookahead nascosto dietro una cache sull'indice
    k = (len(df), df.index[0], df.index[-1])
    if k not in _CACHE_TRAPPOLA:
        _CACHE_TRAPPOLA[k] = df["High"].groupby(df.index.date).transform("max")
    return df["High"] >= _CACHE_TRAPPOLA[k]

def t_usa_indicatore(df):      # pulita in se', ma legge un indicatore che guarda avanti
    return df["Close"] > df["media_centrata"]

# --- pulite e mai vera ----------------------------------------------------
def t_pulita_evento(df):
    c, e = df["Close"], df["ema20"]
    return _evento((c > e) & (c.shift(1) <= e.shift(1)))

def t_pulita_filtro(df):
    return df["Close"] > df["Close"].rolling(50).mean()

def t_mai_vera(df):
    return df["Close"] < 0

def prepara_con_trappola(df):
    d = aggiungi_indicatori(df)
    d["media_centrata"] = d["Close"].rolling(9, center=True).mean()
    return d

R.register_entry("T_SHIFT_MENO_UNO", 1)(t_shift_meno_uno)
R.register_entry("T_SHIFT_RARO", 1)(t_shift_raro)
R.register_entry("T_PULITA_EVENTO", 1)(t_pulita_evento)
for nome, f in [("T_MASSIMO_GIORNO", t_massimo_giorno), ("T_ROLLING_CENTRATA", t_rolling_centrata),
                ("T_QUANTILE_GLOBALE", t_quantile_globale), ("T_PROSSIMO_TIMESTAMP", t_prossimo_timestamp),
                ("T_CACHE", t_cache), ("T_USA_INDICATORE", t_usa_indicatore),
                ("T_PULITA_FILTRO", t_pulita_filtro), ("T_MAI_VERA", t_mai_vera)]:
    R.register_filter(nome)(f)

ATTESI_A1 = {
    "T_SHIFT_MENO_UNO": "LOOKAHEAD", "T_SHIFT_RARO": "LOOKAHEAD",
    "T_MASSIMO_GIORNO": "LOOKAHEAD", "T_ROLLING_CENTRATA": "LOOKAHEAD",
    "T_QUANTILE_GLOBALE": "LOOKAHEAD", "T_PROSSIMO_TIMESTAMP": "LOOKAHEAD",
    "T_CACHE": "LOOKAHEAD", "T_USA_INDICATORE": "LOOKAHEAD",
    "media_centrata": "LOOKAHEAD",
    "T_PULITA_EVENTO": "OK", "T_PULITA_FILTRO": "OK", "T_MAI_VERA": "NON_ESERCITATA",
    # gli indicatori veri devono uscire puliti
    **{c: "OK" for c in ("rsi", "macd", "macd_signal", "macd_hist", "ema20", "ema50",
                         "zlema50", "atr", "realized_vol", "adx", "vwap")},
}

rep_a1 = verifica_lookahead(mercato_sintetico(), prepara=prepara_con_trappola, verbose=False)
ottenuti = rep_a1.set_index("nome")["esito"]
verifica_a1 = pd.DataFrame({"atteso": pd.Series(ATTESI_A1),
                            "ottenuto": ottenuti.reindex(list(ATTESI_A1))})
verifica_a1["giusto"] = verifica_a1["atteso"] == verifica_a1["ottenuto"]
display(verifica_a1)
giusti = int(verifica_a1["giusto"].sum())
print(f"SEZIONE A1: {giusti}/{len(verifica_a1)}")
assert giusti == len(verifica_a1), "Il test di lookahead NON riconosce le trappole: le sezioni B e C non valgono."

## A2 · Collaudo del test di degenerazione

Condizioni costruite a mano, con risultato noto: sempre vera, sempre falsa, quasi sempre vera, rara, uno *stato* al posto di un evento, una normale, una a grappoli, una che non restituisce booleani, una che va in errore, e l'exit «nessuna uscita» che deve risultare `ATTESA`.

Per l'entry a grappoli il risultato è noto esattamente: gruppi di 4 eventi a 2 barre di distanza, quindi `quota_in_grappolo` = 0,75. Con `distanza_grappolo=1` gli stessi eventi risultano tutti separati (quota 0).

In [ ]:
R.clear_registry()
df_a2 = aggiungi_indicatori(mercato_sintetico())
N = len(df_a2)

def maschera(df, posizioni):
    m = np.zeros(len(df), dtype=bool)
    m[np.asarray(list(posizioni), dtype=int)] = True
    return pd.Series(m, index=df.index)

GRUPPI = range(100, N - 10, 200)          # un grappolo ogni 200 barre
R.register_filter("D_SEMPRE_VERA")(lambda df: pd.Series(True, index=df.index))
R.register_filter("D_SEMPRE_FALSA")(lambda df: pd.Series(False, index=df.index))
R.register_filter("D_QUASI_SEMPRE")(lambda df: ~maschera(df, range(0, len(df), 50)))   # 98%
R.register_filter("D_NON_BOOLEANA")(lambda df: df["Close"])
R.register_filter("D_ERRORE")(lambda df: df["colonna_che_non_esiste"] > 0)
R.register_entry("D_RARA", 1)(lambda df: maschera(df, range(500, 500 + 10 * 300, 300)))   # 10 eventi
R.register_entry("D_STATO", 1)(lambda df: maschera(df, [p + k for p in range(0, len(df) - 3, 100) for k in range(3)]))
R.register_entry("D_OK", 1)(lambda df: maschera(df, range(0, len(df), 50)))
R.register_entry("D_GRAPPOLO", 1)(lambda df: maschera(df, [g + k for g in GRUPPI for k in (0, 2, 4, 6)]))
R.register_exit("X0_NO_EXIT", 1)(lambda df: pd.Series(False, index=df.index))

rep_a2 = verifica_degenerazione(df_a2, verbose=False).set_index("nome")
rep_a2_d1 = verifica_degenerazione(df_a2, distanza_grappolo=1, verbose=False).set_index("nome")

ATTESI_A2 = {
    "D_SEMPRE_VERA": "SEMPRE_VERA", "D_SEMPRE_FALSA": "SEMPRE_FALSA",
    "D_QUASI_SEMPRE": "QUASI_SEMPRE_VERA", "D_NON_BOOLEANA": "NON_BOOLEANA",
    "D_ERRORE": "ERRORE", "D_RARA": "RARA", "D_STATO": "STATO",
    "D_OK": "OK", "D_GRAPPOLO": "OK", "X0_NO_EXIT": "ATTESA",
}
verifica_a2 = pd.DataFrame({"atteso": pd.Series(ATTESI_A2),
                            "ottenuto": rep_a2["esito"].reindex(list(ATTESI_A2))})
controlli = {
    "D_GRAPPOLO: grappoli = numero di gruppi": rep_a2.loc["D_GRAPPOLO", "grappoli"] == len(GRUPPI),
    "D_GRAPPOLO: quota_in_grappolo = 0,75": abs(rep_a2.loc["D_GRAPPOLO", "quota_in_grappolo"] - 0.75) < 1e-12,
    "D_GRAPPOLO con distanza 1: quota 0": rep_a2_d1.loc["D_GRAPPOLO", "quota_in_grappolo"] == 0,
    "D_OK: nessun grappolo": rep_a2.loc["D_OK", "quota_in_grappolo"] == 0,
    "D_RARA: 10 eventi": rep_a2.loc["D_RARA", "eventi"] == 10,
    "D_STATO: barre consecutive contate": rep_a2.loc["D_STATO", "barre_consecutive"] == 2 * rep_a2.loc["D_STATO", "eventi"],
}
for nome, ok in controlli.items():
    verifica_a2.loc[nome] = ["vero", "vero" if ok else "falso"]
verifica_a2["giusto"] = verifica_a2["atteso"] == verifica_a2["ottenuto"]
display(verifica_a2)
giusti = int(verifica_a2["giusto"].sum())
print(f"SEZIONE A2: {giusti}/{len(verifica_a2)}")
assert giusti == len(verifica_a2), "Il test di degenerazione NON classifica bene i casi noti."

## B · Lookahead su tutto il catalogo

Dati sintetici (quattro mesi di random walk M15 con i cambi d'ora d'autunno). Per ogni condizione: 104 tagli comuni (uno per ogni quarto d'ora, più i bordi del weekend) e fino a 5 tagli mirati sulle barre in cui la condizione è vera. Tempo: circa 2 minuti.

**Come si legge**
- `LOOKAHEAD` → la condizione cambia valore quando si toglie il futuro. La colonna `primo_caso` dice dove.
- `ERRORE` → la condizione non gira; il messaggio è nella colonna `errore`.
- `NON_ESERCITATA` → mai vera su questi dati, quindi non verificabile. Le due exit `X0_…_NO_EXIT` sono sempre false per costruzione: per loro è normale.
- Le righe `indicatore` sono gli indicatori di `engine/indicatori.py`, verificati da soli.

Il test fallisce solo per `LOOKAHEAD` o `ERRORE`.

In [ ]:
registra_catalogo()
rep_b = verifica_lookahead(mercato_sintetico())
problemi_b = rep_b[rep_b["esito"].isin(["LOOKAHEAD", "ERRORE"])]
print(f"\nSEZIONE B: {'SUPERATA' if problemi_b.empty else 'NON SUPERATA'} — "
      f"{len(problemi_b)} condizioni con LOOKAHEAD o ERRORE")
display(rep_b[rep_b["esito"] != "OK"])

La tabella completa, tutte le righe:

In [ ]:
rep_b

## C · Degenerazione sui dati veri

EURUSD M15 dal repo, convertito in UTC con la stessa regola del notebook principale, indicatori da `engine/indicatori.py`. Nessun promosso/bocciato: la degenerazione dipende dal dataset.

**Come si legge**
- `SEMPRE_FALSA` → mai vera: di solito è un bug (colonna sbagliata, soglia impossibile).
- `SEMPRE_VERA` / `QUASI_SEMPRE_VERA` → vera sul 95% delle barre o più: non filtra.
- `RARA` → meno di 30 occorrenze (per le entry si contano gli eventi, per filtri ed exit le barre vere).
- `STATO` → un'entry vera su barre consecutive: manca `_evento`.
- `ATTESA` → le exit «nessuna uscita», sempre false per costruzione.
- `grappoli` / `quota_in_grappolo` (solo entry) → quanti eventi cadono entro `distanza_grappolo` barre dal precedente. Informativo, non cambia l'esito.

Le soglie si cambiano nella chiamata: `verifica_degenerazione(df_vero, min_occorrenze=30, copertura_max=0.95, distanza_grappolo=8)`.

In [ ]:
df_vero = pd.read_csv("EURUSD_M15.csv", index_col="Date")
df_vero = to_utc_index(df_vero, "A_US_DST (NY+7h)", on_dst_gap="drop", verbose=False)
df_vero = aggiungi_indicatori(df_vero)

rep_c = verifica_degenerazione(df_vero, min_occorrenze=30, copertura_max=0.95, distanza_grappolo=8)
display(rep_c[rep_c["esito"] != "OK"])

La tabella completa, ordinata per tipo e copertura:

In [ ]:
rep_c.sort_values(["tipo", "copertura"]).reset_index(drop=True)

## D · I comportamenti di backtesting.py su cui si regge il tool

Tre cose che il tool dà per scontate, verificate su una serie di 10 barre costruita a mano con prezzi noti, un trade solo, **usando la stessa strategia dell'engine** (`_StrategiaGenerica`):

- **D1 · commissione** — fuori dal prezzo di fill, addebitata su **entrambi** i lati. E `avg_trade_netto` dell'engine torna al conto fatto a mano.
- **D2 · spread** — **una volta sola**, dentro il prezzo d'ingresso (long a Open × (1+s), short a Open × (1−s)); l'uscita resta al prezzo pieno.
- **D3 · stop** — assegnato come fa l'engine, **non è attivo sulla barra d'ingresso** e scatta dalla barra dopo.

In fondo: `SEZIONE D: n/n`. Se non passa, il notebook si ferma: una versione diversa di `backtesting.py` ha cambiato un comportamento, e i numeri dei backtest non si leggono più come prima.

In [ ]:
import backtesting
from backtesting import Backtest
from engine.exit_search_bt import _StrategiaGenerica
from engine.metriche import pips_per_trade

# --- una serie di 10 barre costruita a mano, prezzi noti ------------------
# Il segnale scatta sulla barra 1: l'engine entra all'Open della barra 2.
# Con n_barre=3 chiede la chiusura sulla barra 4 ed esce all'Open della 5.
OPEN = np.array([1.1000, 1.1000, 1.1010, 1.1020, 1.1030, 1.1040, 1.1050, 1.1060, 1.1070, 1.1080])
PIP = 0.0001

def serie_prova(stop_frac=np.nan, minimo_barra2=None):
    idx = pd.date_range("2024-01-02 10:00", periods=len(OPEN), freq="15min", tz="UTC")
    d = pd.DataFrame({"Open": OPEN, "High": OPEN + 0.0005, "Low": OPEN - 0.0005,
                      "Close": OPEN + 0.0002, "Volume": 100.0}, index=idx)
    if minimo_barra2 is not None:
        d.iloc[2, d.columns.get_loc("Low")] = minimo_barra2
    segnale = np.zeros(len(d), dtype=bool); segnale[1] = True
    d["__segnale"] = segnale
    for lato in ("long", "short"):
        d[f"sl_{lato}"] = stop_frac
        d[f"tp_{lato}"] = np.nan
    return d

def un_trade(d, lato, spread=0.0, commission=0.0, n_barre=3):
    bt = Backtest(d, _StrategiaGenerica, cash=100_000, spread=spread,
                  commission=commission, exclusive_orders=True, finalize_trades=True)
    kw = {"long_col": "__segnale", "short_col": None} if lato == "long" else \
         {"long_col": None, "short_col": "__segnale"}
    st = bt.run(n_barre=n_barre, exit_rule_long_col=None, exit_rule_short_col=None, **kw)
    return st["_trades"]

prove = {}

# --- D1 · commissione: fuori dal prezzo, su entrambi i lati --------------
c = 0.0001
for lato in ("long", "short"):
    t = un_trade(serie_prova(), lato, commission=c)
    r = t.iloc[0]
    prezzi_puri = np.isclose(r.EntryPrice, OPEN[2]) and np.isclose(r.ExitPrice, OPEN[5])
    attesa = c * (OPEN[2] + OPEN[5]) * abs(r.Size)           # entrata + uscita
    addebitata = np.isclose(r.Commission, attesa)
    lordi, netti = pips_per_trade(t, PIP, c)
    segno = 1 if lato == "long" else -1
    netto_a_mano = segno * (OPEN[5] - OPEN[2]) / PIP - c * (OPEN[2] + OPEN[5]) / PIP
    prove[f"D1 {lato}: prezzi di fill senza commissione"] = prezzi_puri
    prove[f"D1 {lato}: commissione = c x (entrata + uscita)"] = addebitata
    prove[f"D1 {lato}: avg_trade_netto dell'engine = conto a mano"] = bool(np.isclose(netti.iloc[0], netto_a_mano))

# --- D2 · spread: una volta sola, dentro il prezzo d'ingresso -----------
s = 0.0002
for lato, segno in (("long", 1), ("short", -1)):
    r = un_trade(serie_prova(), lato, spread=s).iloc[0]
    prove[f"D2 {lato}: ingresso = Open x (1 {'+' if segno > 0 else '-'} spread)"] = bool(np.isclose(r.EntryPrice, OPEN[2] * (1 + segno * s)))
    prove[f"D2 {lato}: uscita al prezzo pieno (spread non ripetuto)"] = bool(np.isclose(r.ExitPrice, OPEN[5]))

# --- D3 · stop: non attivo sulla barra d'ingresso ------------------------
# Stop al 20% sotto l'ingresso (long). La barra d'ingresso (2) ha un minimo
# che lo buca: se lo stop fosse attivo li', il trade uscirebbe sulla barra 2.
# Deve invece restare aperto e uscire a tempo, all'Open della barra 5.
stop = 0.20
buco = OPEN[2] * (1 - stop) - 0.01
r = un_trade(serie_prova(stop_frac=stop, minimo_barra2=buco), "long").iloc[0]
prove["D3: lo stop NON scatta sulla barra d'ingresso"] = bool(r.ExitBar != 2)
# controllo: lo stesso buco sulla barra dopo l'ingresso (3) lo fa scattare
d = serie_prova(stop_frac=stop)
d.iloc[3, d.columns.get_loc("Low")] = buco
r3 = un_trade(d, "long").iloc[0]
prove["D3: lo stop scatta dalla barra successiva"] = bool(r3.ExitBar == 3 and np.isclose(r3.ExitPrice, r3.EntryPrice * (1 - stop)))

verifica_d = pd.DataFrame({"superata": pd.Series(prove)})
display(verifica_d)
giusti = int(verifica_d["superata"].sum())
print(f"backtesting.py {backtesting.__version__}")
print(f"SEZIONE D: {giusti}/{len(verifica_d)}")
assert giusti == len(verifica_d), ("backtesting.py non si comporta piu' come il tool si aspetta: "
                                    "i numeri dei backtest non vanno letti finche' non si chiarisce.")

## E · Le soglie di rumore

La tabella dell'event study e quella delle uscite hanno una colonna `oltre_rumore`: **True** quando il risultato è abbastanza alto da non essere spiegabile col caso, tenendo conto di quante prove si sono fatte.

Qui si controlla che quella colonna dica il vero. Prezzi casuali, entry casuali: nessuna ha niente da dire, quindi la soglia deve scattare **per sbaglio circa 5 volte su 100**, non di più.

- **E1** — con orizzonte 1 la soglia nuova coincide con quella di sempre.
- **E2** — con trigger distanti l'incertezza dell'event study è la formula classica.
- **E3–E5** — falsi allarmi dell'event study con trigger distanti, frequenti (uno ogni ~25 barre) e a grappoli (4 in 8 candele). Accettati fra 1% e 10%.
- **E6** — la `t` della ricerca delle uscite è tarata.

Senza la correzione dell'incertezza del 24/9 E4 e E5 fallirebbero: 14% e 93% di falsi allarmi.

In fondo: `SEZIONE E: n/n`. Tempo: circa 3 minuti.

In [ ]:
from engine.giudizio import soglia_rumore, soglia_rumore_orizzonte
from engine.event_study import run_event_study
from engine.exit_search_bt import run_exit_search_bt

H = 25
prove_e, misure_e = {}, {}

# --- E1 · con orizzonte 1 la soglia nuova e' quella di sempre -----------
prove_e["E1: soglia con orizzonte 1 = soglia_rumore(k)"] = all(
    soglia_rumore_orizzonte(k, 1) == soglia_rumore(k) for k in (1, 5, 17, 52))

# --- E2 · senza sovrapposizioni l'incertezza e' la formula classica -----
df_e = mercato_sintetico(seme=7)
R.clear_registry()
sparsa = np.zeros(len(df_e), dtype=bool); sparsa[5::60] = True     # un trigger ogni 60 barre
R.register_entry("SPARSA", 1)(lambda d: pd.Series(sparsa, index=d.index))
with contextlib.redirect_stdout(io.StringIO()):
    ev = run_event_study(df_e, horizon=H, min_trades=0)
pos = np.flatnonzero(sparsa)
ingresso = df_e["Open"].to_numpy()[pos + 1]
classica = []
for k in range(1, H + 1):
    ok = pos + k < len(df_e)
    r = df_e["Close"].to_numpy()[pos[ok] + k] / ingresso[ok] - 1.0
    classica.append(r.std(ddof=1) / np.sqrt(r.size) * 100)
prove_e["E2: trigger distanti -> incertezza = deviazione std / radice(n)"] = bool(
    np.allclose(ev.incertezza["SPARSA"].to_numpy(), classica, rtol=1e-10))

# --- E3-E5 · falsi allarmi dell'event study su entry casuali ------------
# Prezzi casuali, 20 entry casuali: nessuna ha niente da dire. La soglia
# deve scattare per sbaglio circa 5 volte su 100.
def entry_casuale(n, rng, modo):
    m = np.zeros(n, dtype=bool)
    if modo == "distanti":            # un trigger ogni 27-56 barre
        p = int(rng.integers(0, 50))
        while p < n - H - 2:
            m[p] = True; p += H + 2 + int(rng.integers(0, 30))
    elif modo == "frequenti":         # in media uno ogni 25 barre
        m = rng.random(n) < 1 / 25
        m &= ~np.r_[False, m[:-1]]
    else:                             # grappoli: 4 trigger in 8 candele
        centri = rng.choice(np.arange(10, n - H - 20), size=40, replace=False)
        for o in (0, 2, 4, 6):
            m[centri + o] = True
    m[-H - 2:] = False
    return m

GIRI = 300
for modo, nome in (("distanti", "E3"), ("frequenti", "E4"), ("grappoli", "E5")):
    allarmi = 0
    for giro in range(GIRI):
        rng = np.random.default_rng(1000 + giro)
        df_g = mercato_sintetico(seme=giro)
        R.clear_registry()
        for j in range(20):
            m = entry_casuale(len(df_g), rng, modo)
            R.register_entry(f"R{j}", 1)(lambda d, m=m: pd.Series(m, index=d.index))
        with contextlib.redirect_stdout(io.StringIO()):
            ev = run_event_study(df_g, horizon=H, min_trades=0)
        allarmi += int(ev.sintesi["volte_incertezza"].abs().max() > ev.soglia_rumore)
    quota = allarmi / GIRI
    misure_e[f"{nome}: falsi allarmi event study, trigger {modo}"] = f"{quota:.1%}"
    prove_e[f"{nome}: falsi allarmi event study, trigger {modo} (atteso ~5%)"] = 0.01 <= quota <= 0.10
    print(f"{nome} · trigger {modo:<9}: falsi allarmi {allarmi}/{GIRI} = {quota:.1%}")

# --- E6 · la t della ricerca delle uscite e' tarata ----------------------
# 150 giri x 5 entry casuali, uscita a tempo, costi zero: su rumore puro
# |t| > 1,96 deve capitare circa il 5% delle volte.
tutte_t, allarmi = [], 0
for giro in range(150):
    rng = np.random.default_rng(500 + giro)
    df_g = mercato_sintetico(seme=10_000 + giro)
    R.clear_registry(); nomi = []
    for j in range(5):
        m = rng.random(len(df_g)) < 1 / 40; m[-12:] = False
        R.register_entry(f"C{j}", 1)(lambda d, m=m: pd.Series(m, index=d.index)); nomi.append(f"C{j}")
    with contextlib.redirect_stdout(io.StringIO()):
        es = run_exit_search_bt(df_g, entry_cols_long=nomi, n_barre=10,
                                spread=0.0, commission=0.0, min_trades=0)
    tutte_t += list(es.risultati["t_stat"])
    allarmi += int(es.risultati["oltre_rumore"].any())
quota_t = float((np.abs(tutte_t) > 1.96).mean())
misure_e["E6: |t| > 1,96 su rumore puro"] = f"{quota_t:.1%} di {len(tutte_t)}"
misure_e["E6: falsi allarmi ricerca uscite (migliore di 5)"] = f"{allarmi / 150:.1%}"
prove_e["E6: t delle uscite tarata (|t| > 1,96 fra 2,5% e 8%)"] = 0.025 <= quota_t <= 0.08
print(f"E6 · |t| > 1,96 nel {quota_t:.1%} di {len(tutte_t)} righe · "
      f"falsi allarmi sul migliore di 5: {allarmi}/150")

R.clear_registry()
verifica_e = pd.DataFrame({"superata": pd.Series(prove_e)})
display(verifica_e)
giusti = int(verifica_e["superata"].sum())
print(f"SEZIONE E: {giusti}/{len(verifica_e)}")
assert giusti == len(verifica_e), "Le soglie di rumore non sono tarate: la colonna oltre_rumore non va letta."